# Topic Modeling with BERTopic

**Course:** [Natural Language Processing](https://ml-viz-ruby.vercel.app/courses/nlp/06-topic-modeling-bertopic)

**The idea in one sentence.** Given a pile of unlabelled documents, discover the
themes hiding in them — BERTopic does it as a **four-stage pipeline** where each
stage is a standard tool you already know:

$$\underbrace{\text{embed}}_{\text{Sentence-BERT}} \;\to\;
\underbrace{\text{reduce}}_{\text{UMAP}} \;\to\;
\underbrace{\text{cluster}}_{\text{HDBSCAN}} \;\to\;
\underbrace{\text{label}}_{\text{c-TF-IDF}}$$

**Why this beats classic LDA.** LDA models topics as bag-of-words mixtures and
never sees word *meaning*. BERTopic clusters *contextual embeddings*, so "refund"
and "reimbursement" land together even though they share no characters — then
labels each cluster with the words that make it distinctive (class-based TF-IDF).

We build all four stages from scratch on a toy support-ticket corpus (random
projection for embeddings, NumPy PCA, NumPy KMeans, c-TF-IDF), **validate our
clustering against scikit-learn**, and close with gotchas + exercises.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND = '#6366f1'
TEAL  = '#2dd4bf'
ROSE  = '#fb7185'
ORANGE = '#f97316'
MUTED = '#475569'
TOPIC_COLORS = [BRAND, TEAL, ROSE]

## 1. A toy support-ticket corpus

We'll hand-craft 60 short documents that fall into three obvious topics:

- **billing**: refund / invoice / charge / payment
- **shipping**: delivery / tracking / shipment / late
- **quality**: defective / broken / faulty / damaged

Each document mixes 4 topic-specific words with 3 filler words shared across topics (`the`, `is`, `my`, etc.). This is small enough to read by eye but large enough to make c-TF-IDF non-trivial.

In [ ]:
TOPIC_VOCAB = {
    'billing':  ['refund', 'invoice', 'charge',   'payment',  'card',   'bill'],
    'shipping': ['delivery','tracking','shipment', 'late',     'address','package'],
    'quality':  ['defective','broken','faulty',    'damaged',  'return', 'product'],
}
FILLER = ['the','is','my','was','i','this','order','please','need','received']
TOPIC_NAMES = list(TOPIC_VOCAB.keys())

def make_doc(rng, topic):
    body = list(rng.choice(TOPIC_VOCAB[topic], size=4, replace=True))
    filler = list(rng.choice(FILLER, size=3, replace=True))
    words = body + filler
    rng.shuffle(words)
    return ' '.join(words)

rng = np.random.default_rng(7)
true_labels = []
docs = []
for topic in TOPIC_NAMES:
    for _ in range(20):  # 20 docs per topic -> 60 total
        docs.append(make_doc(rng, topic))
        true_labels.append(topic)
true_labels = np.array(true_labels)

print(f'{len(docs)} documents, {len(set(true_labels))} ground-truth topics')
for i in [0, 1, 20, 21, 40, 41]:
    print(f'  [{true_labels[i]:<8}] {docs[i]}')

## 2. Stage 1 — Embed (simulated Sentence-BERT)

Real BERTopic uses Sentence-BERT to encode each document into a 384- or 768-dimensional vector. To keep this notebook self-contained and deterministic, we'll *simulate* that step: build a bag-of-words vector for each document, then multiply by a **frozen random projection matrix**. Random projections preserve pairwise distances (Johnson–Lindenstrauss), so semantically similar documents (same topic vocab) end up with similar projected vectors.

This is not a real sentence encoder, but it is enough to demonstrate the pipeline.

In [ ]:
# Build vocabulary across the full corpus
vocab = sorted({w for d in docs for w in d.split()})
word_to_idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f'Vocabulary size: {V}')

# Bag-of-words matrix: n_docs x V
def bow(doc):
    v = np.zeros(V)
    for w in doc.split():
        v[word_to_idx[w]] += 1.0
    return v

X_bow = np.stack([bow(d) for d in docs])
print(f'BoW shape: {X_bow.shape}')

# Frozen random-projection matrix V -> 64 (simulated embedding dim)
EMBED_DIM = 64
proj_rng = np.random.default_rng(123)
PROJ = proj_rng.normal(0, 1.0 / np.sqrt(EMBED_DIM), size=(V, EMBED_DIM))

X_emb = X_bow @ PROJ                            # shape: (n_docs, EMBED_DIM)
X_emb = X_emb / np.linalg.norm(X_emb, axis=1, keepdims=True)  # L2-normalise like SBERT
print(f'Simulated embedding shape: {X_emb.shape}')
print(f'First doc embedding norm: {np.linalg.norm(X_emb[0]):.3f}')

## 3. Stage 2 — Reduce (NumPy PCA to 2D for inspection)

Real BERTopic reduces to roughly 5 dimensions with UMAP. For this toy demo we go straight to 2D with PCA so we can scatter-plot the result. The scatter plot is a *diagnostic* — if you can already see the topics separating cleanly, the embedding is doing its job.

In [ ]:
def pca(X, n_components):
    Xc = X - X.mean(axis=0, keepdims=True)
    # SVD-based PCA: more numerically stable than eig(cov)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:n_components].T

X_2d = pca(X_emb, n_components=2)
print(f'Reduced shape: {X_2d.shape}')

fig, ax = plt.subplots(figsize=(6.5, 5))
for color, topic in zip(TOPIC_COLORS, TOPIC_NAMES):
    mask = true_labels == topic
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=topic,
               s=60, edgecolor='#0f1117', alpha=0.9)
ax.set_xlabel('PCA dim 1'); ax.set_ylabel('PCA dim 2')
ax.set_title('2D projection of simulated sentence embeddings\n(colours = ground-truth topics)')
ax.legend(loc='best', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

Three clusters separate cleanly in 2D — exactly the picture you want before clustering. In a real BERTopic run this would be the UMAP-2D version; clustering would actually happen on a 5D UMAP projection (more separation, still low-dim enough for density to be meaningful).

## 4. Stage 3 — Cluster (NumPy KMeans, $K = 3$)

Real BERTopic uses HDBSCAN to discover $K$ automatically. For this toy demo we use KMeans with a known $K = 3$ to keep the code short. The point is the *pipeline shape*, not the choice of clusterer.

In [ ]:
def kmeans(X, K, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    # Init: pick K random data points as centroids
    idx = rng.choice(len(X), size=K, replace=False)
    centroids = X[idx].copy()
    for _ in range(n_iter):
        # Assign each point to the nearest centroid
        d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
        assignments = d2.argmin(axis=1)
        # Recompute centroids
        new_centroids = np.stack([
            X[assignments == k].mean(axis=0) if (assignments == k).any() else centroids[k]
            for k in range(K)
        ])
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids
    return assignments, centroids

# Cluster the 2D reduction (in practice you'd cluster a 5D UMAP)
assignments, centroids = kmeans(X_2d, K=3, seed=42)

# Cluster -> ground-truth agreement (greedy match)
from itertools import permutations
best_acc, best_map = 0.0, None
for perm in permutations(range(3)):
    mapped = np.array([TOPIC_NAMES[perm[c]] for c in assignments])
    acc = (mapped == true_labels).mean()
    if acc > best_acc:
        best_acc, best_map = acc, perm
print(f'Best cluster→topic mapping: {[TOPIC_NAMES[i] for i in best_map]}')
print(f'Accuracy: {best_acc:.2%}')

fig, ax = plt.subplots(figsize=(6.5, 5))
for k, c in enumerate(TOPIC_COLORS):
    mask = assignments == k
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=c, label=f'cluster {k}',
               s=60, edgecolor='#0f1117', alpha=0.9)
ax.scatter(centroids[:, 0], centroids[:, 1], c='white', marker='X', s=180,
           edgecolor='#0f1117', linewidths=1.5, label='centroids', zorder=5)
ax.set_xlabel('PCA dim 1'); ax.set_ylabel('PCA dim 2')
ax.set_title(f'KMeans clusters (K=3) on the 2D reduction — accuracy {best_acc:.0%}')
ax.legend(loc='best', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

### Validate: our KMeans matches scikit-learn and recovers the true topics

Two checks on the clustering stage. First, our from-scratch KMeans should produce
the *same partition* as `sklearn.cluster.KMeans` on the same 2D points — cluster
labels are arbitrary, so we compare with the **adjusted Rand index** (1.0 = same
partition, invariant to label permutation). Second, the discovered clusters
should line up with the planted topics (high purity).

In [ ]:
from sklearn.cluster import KMeans as SKKMeans
from sklearn.metrics import adjusted_rand_score

sk_assign = SKKMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_2d)
ari_vs_sklearn = adjusted_rand_score(assignments, sk_assign)
ari_vs_truth   = adjusted_rand_score(true_labels, assignments)
print(f'adjusted Rand index, ours vs sklearn : {ari_vs_sklearn:.3f}  (1.0 = identical partition)')
print(f'adjusted Rand index, ours vs truth   : {ari_vs_truth:.3f}  (topics recovered)')
assert ari_vs_sklearn > 0.9, 'our KMeans should closely match sklearn on this separable toy'
assert ari_vs_truth   > 0.9, 'clustering should recover the planted topics'
print('\n✅ from-scratch KMeans agrees with sklearn and recovers the three topics')

## 5. Stage 4 — Label with class-based TF-IDF

Each cluster is now a set of documents. To turn it into a *topic*, we need a label — typically the top-5 words that best characterise the cluster. **Class-based TF-IDF** treats each cluster's documents as one super-document, then ranks terms by how distinctively they appear in that cluster:

$$
\text{c-TFIDF}(t, c) \;=\; \mathrm{tf}_{t,c} \cdot \log\!\left(1 + \frac{\bar{f}}{f_t}\right)
$$

where $\mathrm{tf}_{t,c}$ is the term frequency inside cluster $c$'s super-document, $f_t$ is the total frequency of $t$ across the corpus, and $\bar{f}$ is the average number of words per cluster.

In [ ]:
def c_tf_idf(docs_by_cluster, vocab):
    """Class-based TF-IDF.

    Args:
        docs_by_cluster: dict cluster_id -> list of token lists
        vocab: sorted list of vocabulary terms

    Returns:
        score: (n_clusters, |vocab|) array of c-TF-IDF scores
    """
    n_clusters = len(docs_by_cluster)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)

    tf = np.zeros((n_clusters, V))
    for c, doc_list in docs_by_cluster.items():
        for tokens in doc_list:
            for w in tokens:
                if w in word_to_idx:
                    tf[c, word_to_idx[w]] += 1

    f_t = tf.sum(axis=0)                          # total freq of each term across corpus
    f_bar = tf.sum(axis=1).mean()                 # avg words per class
    idf_like = np.log(1.0 + f_bar / np.maximum(f_t, 1e-9))
    return tf * idf_like[None, :]

# Group docs by KMeans cluster
docs_by_cluster = {k: [] for k in range(3)}
for i, c in enumerate(assignments):
    docs_by_cluster[int(c)].append(docs[i].split())

scores = c_tf_idf(docs_by_cluster, vocab)
print(f'c-TF-IDF score matrix shape: {scores.shape}')

TOP_K = 5
for c in range(3):
    top_idx = np.argsort(scores[c])[::-1][:TOP_K]
    top_words = [vocab[i] for i in top_idx]
    print(f'Cluster {c}: {top_words}')

The top words per cluster are exactly the topic-specific vocab we planted: `refund/invoice/charge/payment/card` for billing, `delivery/tracking/shipment/late/address` for shipping, `defective/broken/faulty/damaged/return` for quality. Common filler words like `the` and `is` appear with huge term frequency in every cluster, but $\log(1 + \bar{f}/f_t)$ crushes their score to near zero — the IDF analogue doing its job at the cluster axis.

Visualise the top-5 c-TF-IDF scores per cluster:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for c, ax in enumerate(axes):
    top_idx = np.argsort(scores[c])[::-1][:TOP_K]
    words = [vocab[i] for i in top_idx]
    vals  = scores[c, top_idx]
    ax.barh(words[::-1], vals[::-1], color=TOPIC_COLORS[c], edgecolor='#0f1117')
    ax.set_title(f'Cluster {c} — top-{TOP_K} by c-TF-IDF')
    ax.set_xlabel('c-TF-IDF score')
plt.tight_layout(); plt.show()

Four-stage pipeline: done. We embedded with a random projection, reduced with PCA, clustered with KMeans, and labelled with c-TF-IDF. Swap each stage for its production counterpart — Sentence-BERT, UMAP, HDBSCAN — and you have BERTopic.

## Gotchas & tradeoffs

| Stage | Gotcha | Production choice |
|-------|--------|-------------------|
| embed | random projection ≈ distances, but has no *semantics* | real Sentence-BERT captures meaning (synonyms cluster) |
| reduce | cluster in high-D and density is meaningless (curse of dimensionality) | UMAP to ~5D **before** clustering |
| cluster | KMeans forces every doc into a topic and needs $K$ | HDBSCAN finds $K$ itself and emits an **outlier topic (−1)** |
| label | raw TF-IDF favours globally-rare words | **c**-TF-IDF ranks per-cluster distinctiveness |

Demo: why you reduce *before* clustering — in high dimensions all pairwise
distances concentrate, so "nearest centroid" loses meaning (the curse of
dimensionality).

In [ ]:
def distance_spread(X):
    # ratio of (max - min) to mean pairwise distance: shrinks as dims grow
    from itertools import combinations
    idx = np.random.default_rng(0).choice(len(X), size=40, replace=False)
    pts = X[idx]
    d = [np.linalg.norm(pts[i] - pts[j]) for i, j in combinations(range(len(pts)), 2)]
    d = np.array(d)
    return (d.max() - d.min()) / d.mean()

rng_hd = np.random.default_rng(0)
for dim in [2, 10, 100, 1000]:
    pts = rng_hd.standard_normal((200, dim))
    print(f'dim {dim:>4}: (max-min)/mean pairwise distance = {distance_spread(pts):.3f}')
print('\nAs dimensionality grows the spread collapses -> every point is ~equidistant,')
print('so density/centroid-based clustering degrades. That is why BERTopic reduces first.')

## ✏️ Your turn

### Exercise: implement `c_tf_idf_student(docs_by_cluster, vocab)`

Implement class-based TF-IDF yourself, following the formula from the lesson:

$$
\text{c-TFIDF}(t, c) = \mathrm{tf}_{t,c} \cdot \log\!\left(1 + \frac{\bar{f}}{f_t}\right)
$$

The test below builds a tiny three-cluster super-corpus and checks that the top word in cluster 0 is `refund` — i.e. that your scoring correctly down-weights filler words that appear in every cluster.

In [ ]:
def c_tf_idf_student(docs_by_cluster, vocab):
    '''Compute class-based TF-IDF.

    Args:
        docs_by_cluster: dict cluster_id -> list of token lists, e.g.
            {0: [['refund', 'the'], ['refund', 'invoice']],
             1: [['delivery', 'the'], ['tracking', 'late']],
             ...}
        vocab: sorted list of vocabulary terms.

    Returns:
        scores: (n_clusters, |vocab|) numpy array of c-TF-IDF scores.
    '''
    # TODO(you): implement the four steps
    #   1. allocate a (n_clusters, |vocab|) zero matrix `tf`
    #   2. for each cluster c, for each token list, accumulate term counts in tf[c, :]
    #   3. compute f_t = total frequency of each term across the corpus
    #            f_bar = average words per class
    #            idf_like = log(1 + f_bar / max(f_t, 1e-9))
    #   4. return tf * idf_like[None, :]
    pass

# Quick smoke run
demo_vocab = ['refund', 'invoice', 'delivery', 'tracking', 'defective', 'broken', 'the', 'is']
demo_clusters = {
    0: [['refund', 'invoice', 'the'], ['refund', 'the', 'is'], ['refund', 'invoice', 'is']],
    1: [['delivery', 'tracking', 'the'], ['delivery', 'the', 'is'], ['tracking', 'is', 'the']],
    2: [['defective', 'broken', 'the'], ['defective', 'the', 'is'], ['broken', 'defective', 'is']],
}
out = c_tf_idf_student(demo_clusters, demo_vocab)
if out is not None:
    print('Top word per cluster:')
    for c in range(3):
        top = demo_vocab[int(np.argmax(out[c]))]
        print(f'  cluster {c}: {top}')

In [ ]:
# Tests
demo_vocab = ['refund', 'invoice', 'delivery', 'tracking', 'defective', 'broken', 'the', 'is']
demo_clusters = {
    0: [['refund', 'invoice', 'the'], ['refund', 'the', 'is'], ['refund', 'invoice', 'is']],
    1: [['delivery', 'tracking', 'the'], ['delivery', 'the', 'is'], ['tracking', 'is', 'the']],
    2: [['defective', 'broken', 'the'], ['defective', 'the', 'is'], ['broken', 'defective', 'is']],
}
scores_s = c_tf_idf_student(demo_clusters, demo_vocab)
assert scores_s is not None, 'Should return an array, not None'
scores_s = np.asarray(scores_s)
assert scores_s.shape == (3, len(demo_vocab)), f'Expected (3, {len(demo_vocab)}), got {scores_s.shape}'

# Top word in cluster 0 should be `refund` (topic-specific, high tf, low cross-cluster freq).
top0 = demo_vocab[int(np.argmax(scores_s[0]))]
assert top0 == 'refund', f'Expected top word in cluster 0 to be "refund", got "{top0}"'

# Top word in cluster 1 should be `delivery` or `tracking` — both shipping-specific.
top1 = demo_vocab[int(np.argmax(scores_s[1]))]
assert top1 in {'delivery', 'tracking'}, f'Expected shipping word in cluster 1, got "{top1}"'

# Top word in cluster 2 should be `defective` or `broken` — both quality-specific.
top2 = demo_vocab[int(np.argmax(scores_s[2]))]
assert top2 in {'defective', 'broken'}, f'Expected quality word in cluster 2, got "{top2}"'

# Filler words `the` and `is` must NOT be the top word in any cluster.
for c in range(3):
    top = demo_vocab[int(np.argmax(scores_s[c]))]
    assert top not in {'the', 'is'}, f'Filler word "{top}" should not top cluster {c}'

# Edge case: a cluster containing an empty document (no tokens) must not crash
# and must not contribute any spurious counts.
edge_clusters = {
    0: [['refund', 'refund'], []],
    1: [['delivery']],
}
edge_vocab = ['refund', 'delivery']
edge_scores = np.asarray(c_tf_idf_student(edge_clusters, edge_vocab))
assert edge_scores.shape == (2, 2)
assert not np.isnan(edge_scores).any(), "An empty document inside a cluster must not produce NaNs"

# Edge case: a single-word "corpus" (one cluster, one document, one token) should
# still produce a finite, well-defined score rather than dividing by zero.
tiny_clusters = {0: [['refund']]}
tiny_scores = np.asarray(c_tf_idf_student(tiny_clusters, ['refund']))
assert tiny_scores.shape == (1, 1)
assert np.isfinite(tiny_scores).all(), "A single-word corpus must not produce inf/NaN"

print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def c_tf_idf_student(docs_by_cluster, vocab):
    n_clusters = len(docs_by_cluster)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)

    tf = np.zeros((n_clusters, V))
    for c, doc_list in docs_by_cluster.items():
        for tokens in doc_list:
            for w in tokens:
                if w in word_to_idx:
                    tf[c, word_to_idx[w]] += 1

    f_t   = tf.sum(axis=0)            # total term frequency across the corpus
    f_bar = tf.sum(axis=1).mean()     # average words per class
    idf_like = np.log(1.0 + f_bar / np.maximum(f_t, 1e-9))
    return tf * idf_like[None, :]
```

Three subtle points:

1. The IDF analogue is computed across **clusters**, not documents. That is the whole point of c-TF-IDF — a word like `the` that appears in every cluster gets $f_t$ ≈ corpus total, so $\bar f / f_t$ is small and $\log(1 + \dots)$ is near zero.
2. The $+1$ inside the log keeps scores non-negative even when $f_t > \bar f$ (which happens for very common words).
3. `np.maximum(f_t, 1e-9)` guards against divide-by-zero if a vocab term never appears in any document of the clusters you passed in.
</details>

---
## Extra practice — DML-sourced retrieval building blocks

c-TF-IDF above scores terms per *cluster*. The two classic retrieval scoring functions below score terms per *document* against a *query* — the building blocks BERTopic's c-TF-IDF step generalizes from. Both are from [Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem).

### Extra practice — DML #60: TF-IDF for a query against a corpus

Uses the same smoothed-IDF trick scikit-learn uses:
$\mathrm{idf}(t) = \log\!\big(\frac{N+1}{\mathrm{df}(t)+1}\big) + 1$, so a term absent from the whole corpus still gets a finite, nonzero weight instead of blowing up or vanishing.

In [ ]:
def compute_tf_idf(corpus, query):
    """
    DML #60 -- smoothed TF-IDF of each query word against each document.

    Args:
        corpus: list of documents, each a list of word tokens
        query: list of word tokens to score
    Returns:
        list of lists: tf_idf[doc_idx][query_idx], rounded to 5 decimals.
    """
    if not corpus:
        return []

    vocab = sorted(set(w for doc in corpus for w in doc) | set(query))
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    N, V = len(corpus), len(vocab)

    tf = np.zeros((N, V))
    for i, doc in enumerate(corpus):
        if not doc:
            continue  # an empty document stays all-zero rather than dividing by zero
        for w in doc:
            tf[i, word_to_idx[w]] += 1
        # TODO(you): normalize row i by document length
        tf[i] /= ...

    # TODO(you): document frequency of each vocab term (# docs containing it at all)
    df = ...

    # TODO(you): smoothed IDF: log((N+1)/(df+1)) + 1
    idf = ...

    tf_idf = tf * idf
    query_idx = [word_to_idx[w] for w in query]
    return np.round(tf_idf[:, query_idx], 5).tolist()


corpus_60 = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "chased", "the", "cat"],
    ["the", "bird", "flew", "over", "the", "mat"],
]
print(compute_tf_idf(corpus_60, ["cat"]))

In [ ]:
# Checks — run me (DML's own test cases)
assert compute_tf_idf(corpus_60, ["cat"]) == [[0.21461], [0.25754], [0.0]]
assert compute_tf_idf(corpus_60, ["cat", "mat"]) == [[0.21461, 0.21461], [0.25754, 0.0], [0.0, 0.21461]]

corpus_60b = [
    ["this", "is", "a", "sample"],
    ["this", "is", "another", "example"],
    ["yet", "another", "sample", "document"],
    ["one", "more", "document", "for", "testing"],
]
assert compute_tf_idf(corpus_60b, ["sample", "document", "test"]) == \
    [[0.37771, 0.0, 0.0], [0.0, 0.0, 0.0], [0.37771, 0.37771, 0.0], [0.0, 0.30217, 0.0]]

# Edge case: an empty document in the corpus must score 0 for the query, not raise
# a ZeroDivisionError.
scores_empty_doc = compute_tf_idf([["cat", "cat", "sat"], []], ["cat"])
assert scores_empty_doc[1] == [0.0], "An empty document should score 0 for any query term"

# Edge case: a single-document, single-word corpus should still produce a finite score.
scores_single = compute_tf_idf([["only"]], ["only"])
assert len(scores_single) == 1 and len(scores_single[0]) == 1

print("✅ DML #60 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compute_tf_idf(corpus, query):
    if not corpus:
        return []
    vocab = sorted(set(w for doc in corpus for w in doc) | set(query))
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    N, V = len(corpus), len(vocab)

    tf = np.zeros((N, V))
    for i, doc in enumerate(corpus):
        if not doc:
            continue
        for w in doc:
            tf[i, word_to_idx[w]] += 1
        tf[i] /= len(doc)

    df = np.count_nonzero(tf > 0, axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1
    tf_idf = tf * idf

    query_idx = [word_to_idx[w] for w in query]
    return np.round(tf_idf[:, query_idx], 5).tolist()
```

The `if not doc: continue` guard is the key robustness fix over a naive port of the formula — without it, an empty document (`len(document) == 0`) would divide by zero when normalizing `tf[doc_idx, :]`.
</details>

### Extra practice — DML #90: BM25 ranking

BM25 is TF-IDF's more careful sibling: it saturates term frequency (diminishing returns for repeated terms via `k1`) and normalizes for document length (`b`) instead of using raw counts. This is the ranking function real search engines — and BERTopic's retrieval-adjacent cousins — actually use.

In [ ]:
def calculate_bm25_scores(corpus, query, k1=1.5, b=0.75):
    """
    DML #90 -- BM25 relevance score of each document in `corpus` against `query`.
    """
    if not corpus or not query:
        raise ValueError("Corpus and query cannot be empty")

    doc_lengths = [len(doc) for doc in corpus]
    avg_doc_length = np.mean(doc_lengths)
    doc_term_counts = [Counter(doc) for doc in corpus]

    doc_freqs = Counter()
    for doc in corpus:
        doc_freqs.update(set(doc))

    N = len(corpus)
    scores = np.zeros(N)

    for term in query:
        # TODO(you): smoothed idf using this term's document frequency
        df = ...
        idf = ...

        for idx, term_counts in enumerate(doc_term_counts):
            if term not in term_counts:
                continue
            tf = term_counts[term]
            # TODO(you): length-normalization factor, then the saturating BM25 term score
            doc_len_norm = ...
            term_score = ...
            scores[idx] += idf * term_score

    return np.round(scores, 3)


print(calculate_bm25_scores([['the', 'cat', 'sat'], ['the', 'dog', 'ran'], ['the', 'bird', 'flew']], ['the', 'cat']))

In [ ]:
# Checks — run me (DML's own test cases)
assert np.allclose(
    calculate_bm25_scores([['the', 'cat', 'sat'], ['the', 'dog', 'ran'], ['the', 'bird', 'flew']], ['the', 'cat']),
    [0.693, 0.0, 0.0], atol=1e-3)
assert np.allclose(calculate_bm25_scores([['the'] * 10, ['the']], ['the']), [0.0, 0.0], atol=1e-3)
assert np.allclose(calculate_bm25_scores([['term'] * 10, ['the'] * 2], ['term'], k1=1.0), [0.705, 0.0], atol=1e-3)

# Edge case: an empty document inside the corpus must never win any relevance score.
scores_empty_doc = calculate_bm25_scores([['cat', 'sat'], []], ['cat'])
assert scores_empty_doc[1] == 0.0, "An empty document should never score positively"

# Edge case: a single-document corpus -> idf = log((1+1)/(1+1)) = 0, so every
# score collapses to 0 regardless of term frequency (every doc "contains" the term
# equally often across a corpus of size 1).
scores_single = calculate_bm25_scores([['only']], ['only'])
assert scores_single[0] == 0.0

# Edge case: an empty corpus must raise, not silently return garbage.
try:
    calculate_bm25_scores([], ['cat'])
    raised = False
except ValueError:
    raised = True
assert raised, "An empty corpus should raise ValueError"

print("✅ DML #90 passed")

<details>
<summary>💡 Show solution</summary>

```python
def calculate_bm25_scores(corpus, query, k1=1.5, b=0.75):
    if not corpus or not query:
        raise ValueError("Corpus and query cannot be empty")

    doc_lengths = [len(doc) for doc in corpus]
    avg_doc_length = np.mean(doc_lengths)
    doc_term_counts = [Counter(doc) for doc in corpus]
    doc_freqs = Counter()
    for doc in corpus:
        doc_freqs.update(set(doc))

    N = len(corpus)
    scores = np.zeros(N)

    for term in query:
        df = doc_freqs.get(term, 0) + 1
        idf = np.log((N + 1) / df)

        for idx, term_counts in enumerate(doc_term_counts):
            if term not in term_counts:
                continue
            tf = term_counts[term]
            doc_len_norm = 1 - b + b * (doc_lengths[idx] / avg_doc_length)
            term_score = (tf * (k1 + 1)) / (tf + k1 * doc_len_norm)
            scores[idx] += idf * term_score

    return np.round(scores, 3)
```
</details>

## Key takeaways

- **BERTopic = embed → reduce → cluster → label**, each stage a swappable
  standard tool (Sentence-BERT, UMAP, HDBSCAN, c-TF-IDF).
- **It clusters meaning, not words.** Contextual embeddings put synonyms
  together — the edge over bag-of-words LDA.
- **Reduce before you cluster.** In high dimensions distances concentrate and
  density-based clustering breaks; UMAP-to-~5D restores separability.
- **c-TF-IDF gives the labels.** Treat each cluster as one super-document and
  rank terms by cluster-level distinctiveness — filler words get crushed.
- We **validated** the clustering against scikit-learn (ARI ≈ 1) and confirmed it
  recovers the planted topics; the DML exercises build the retrieval scorers
  (smoothed TF-IDF, BM25) underneath it all.